In [ ]:
from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('YC_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('YC_SECRET_ACCESS_KEY')
S3_BUCKET   = userdata.get('S3_BUCKET')
S3_ENDPOINT = 'https://storage.yandexcloud.net'

In [ ]:
os.system('pip install ultralytics awscli -q')
os.system(f'aws s3 cp s3://{S3_BUCKET}/second/cv/dataset_detection.tar.gz /content/dataset_detection.tar.gz --endpoint-url {S3_ENDPOINT}')
os.system('tar -xzf /content/dataset_detection.tar.gz -C /content/')
print('dataset ready')

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

RUN    = 'yolov8s_1'
MODEL  = 'yolov8s.pt'
EPOCHS = 50
IMGSZ  = 224
BATCH  = 16
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

CLASSES = ['gun', 'spg', 'ifv', 'uav', 'armored_vehicle', 'apc', 'infantry', 'mlrs', 'tank']

data_yaml = Path('/content/data.yaml')
data_yaml.write_text(
    f'path: /content/dataset_detection\n'
    f'train: images/train\n'
    f'val: images/val\n\n'
    f'nc: {len(CLASSES)}\n'
    f'names: {CLASSES}\n'
)

print(f'device: {DEVICE}  run: {RUN}')

In [ ]:
model = YOLO(MODEL)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project='/content/weights',
    name=RUN,
)

In [ ]:
best = Path(f'/content/weights/{RUN}/weights/best.pt')
os.system(f'aws s3 cp {best} s3://{S3_BUCKET}/second/cv/detection/weights/{RUN}/best.pt --endpoint-url {S3_ENDPOINT}')
print(f'best.pt -> s3/detection/{RUN}')